# Решения. EDA — урок и ДЗ

Эталон к ноутбуку урока и к домашнему заданию. Ученикам до сдачи не показывать. Каждая задача стоит в своей ячейке.


In [ ]:
from pathlib import Path
import pandas as pd


LISTINGS_URL = (
    "https://raw.githubusercontent.com/gurovic/letovo-ml-profile/main/"
    "modules/08_02_carsharing_pandas_lr/data/listings.csv"
)


def find_listings_csv():
    for p in (
        Path("listings.csv"),
        Path("../listings.csv"),
        Path("../../data/listings.csv"),
        Path("../../../data/listings.csv"),
        Path("../data/listings.csv"),
    ):
        if p.exists():
            return p.resolve()
    return LISTINGS_URL


LISTINGS_PATH = find_listings_csv()
df = pd.read_csv(LISTINGS_PATH)
import matplotlib.pyplot as plt


## Урок


## 1. describe: читать таблицу


In [ ]:
stats = df[['accommodates', 'price']].describe()
print(stats)
mean_price = float(df['price'].mean())
print('mean price:', mean_price)

## 2. Среднее и медиана цены


In [ ]:
median_price = float(df['price'].median())
MEAN_MEDIAN_NOTE = (
    'Среднее цены выше медианы: дорогие объявления тянут среднее вправо.'
)
print(mean_price, median_price, MEAN_MEDIAN_NOTE)

## 3. info и пропуски


In [ ]:
n_non_null_price = int(df['price'].notna().sum())
print('non-null price:', n_non_null_price, '/', len(df))

## 4. Scatter: вместимость и цена


In [ ]:
plt.figure(figsize=(6, 4))
plt.scatter(df['accommodates'], df['price'], alpha=0.5)
plt.xlabel('accommodates')
plt.ylabel('price')
plt.title('Цена vs вместимость')
plt.show()
SCATTER_DONE = True

## 5. Что видно на графике


In [ ]:
EDA_CONCLUSION = (
    'С ростом accommodates цена в среднем растёт, но разброс заметный.'
)
print(EDA_CONCLUSION)

## 6. Другой признак: число отзывов


In [ ]:
median_reviews = float(df['number_of_reviews'].median())
plt.figure(figsize=(6, 4))
plt.scatter(df['number_of_reviews'], df['price'], alpha=0.4)
plt.xlabel('number_of_reviews')
plt.ylabel('price')
plt.title('Цена vs отзывы')
plt.show()
print('median reviews:', median_reviews)

## 7. Корреляция — число, не причина


In [ ]:
corr_acc = float(df['accommodates'].corr(df['price']))
CORR_NOTE = (
    'corr — мера линейной согласованности на этих данных, не доказательство причины.'
)
print('corr(accommodates, price):', round(corr_acc, 3))
print(CORR_NOTE)

## 8. Какой признак брать первым


In [ ]:
BETTER_FOR_LINEAR = 'accommodates'
WHY_EDA = (
    'На scatter с accommodates видна наклонная полоса; у number_of_reviews облако более размытое. '
    'Линейная модель на accommodates — первый кандидат.'
)
print(BETTER_FOR_LINEAR, WHY_EDA)

## 9. Сколько объявлений каждого типа комнаты


In [ ]:
room_counts = df['room_type'].value_counts()
top_room = str(room_counts.idxmax())
assert isinstance(room_counts, pd.Series)
assert int(room_counts.sum()) == len(df)
assert top_room in list(room_counts.index)
print(room_counts)
print('top:', top_room)


## 10. Средняя цена внутри типа комнаты


In [ ]:
mean_by_room = df.groupby('room_type')['price'].mean()
assert isinstance(mean_by_room, pd.Series)
assert len(mean_by_room) > 1
assert float(mean_by_room.min()) > 0
print(mean_by_room)


## 11. Сколько разных районов


In [ ]:
n_areas = int(df['neighbourhood'].nunique())
assert n_areas > 1
print('районов:', n_areas)


## 12. Пять самых дорогих объявлений


In [ ]:
top5 = df.sort_values('price', ascending=False).head(5)[
    ['listing_id', 'neighbourhood', 'price']
]
assert top5.shape == (5, 3)
assert list(top5['price']) == sorted(top5['price'], reverse=True)
print(top5)


## 13. Пропуски по каждому столбцу


In [ ]:
missing = df.isna().sum()
assert len(missing) == df.shape[1]
assert int(missing['price']) >= 0
print(missing)


## ДЗ. Районы и число спален


## 1. Сколько объявлений в каждом районе


In [ ]:
area_counts = df['neighbourhood'].value_counts()
top_area = str(area_counts.idxmax())
assert int(area_counts.sum()) == len(df)
assert top_area in list(area_counts.index)
print(area_counts.head())
print('top area:', top_area)


## 2. Медиана цены внутри района


In [ ]:
median_by_area = df.groupby('neighbourhood')['price'].median()
assert len(median_by_area) > 1
assert float(median_by_area.min()) > 0
print(median_by_area.sort_values(ascending=False).head())


## 3. Квартили цены


In [ ]:
q1_price = float(df['price'].quantile(0.25))
q3_price = float(df['price'].quantile(0.75))
assert q1_price < q3_price
print(q1_price, q3_price)


## 4. Сколько объявлений дороже медианы своего района


In [ ]:
area_median = df.groupby('neighbourhood')['price'].transform('median')
n_above_area = int((df['price'] > area_median).sum())
assert 0 < n_above_area < len(df)
print('дороже медианы своего района:', n_above_area)


## 5. Спальни и гости: кто ближе к цене


In [ ]:
corr_bed = float(df['bedrooms'].corr(df['price']))
corr_acc_hw = float(df['accommodates'].corr(df['price']))
CLOSER_TO_PRICE = 'bedrooms' if abs(corr_bed) > abs(corr_acc_hw) else 'accommodates'
WHY_CLOSER = (
    'В линейную модель первым берём признак, который на этих данных '
    'сильнее меняется вместе с ценой; это всё ещё не причина цены.'
)
assert CLOSER_TO_PRICE in ('bedrooms', 'accommodates')
assert len(WHY_CLOSER) > 20
print(round(corr_bed, 3), round(corr_acc_hw, 3), CLOSER_TO_PRICE)
